In [ ]:
import pandas as pd

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from tqdm import tqdm

See COCONUT: https://coconut.naturalproducts.net/download

In [ ]:
coconut = pd.read_csv("/content/coconut_csv-08-2026.csv")

/tmp/ipykernel_1202/4115805187.py:1: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut = pd.read_csv("/content/coconut_csv-08-2026.csv")


In [ ]:
len(coconut)

738827

In [ ]:
training = pd.read_csv("/content/ligands_pdbqt.csv")

In [ ]:
len(training)

5779

In [ ]:
coconut_test = coconut.head(10_000)

In [ ]:
# ============================================================
# TEST: COCONUT TEST SET vs JAK2 QSAR TRAINING SET
# ============================================================

import pandas as pd
import numpy as np
import time

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from tqdm.auto import tqdm


# ============================================================
# 1. CHECK DATA
# ============================================================

print(f"COCONUT test compounds: {len(coconut_test):,}")
print(f"Training compounds: {len(training):,}")

assert "canonical_smiles" in coconut_test.columns
assert "Smiles" in training.columns


# ============================================================
# 2. REMOVE MISSING SMILES
# ============================================================

coconut_test = coconut_test[
    coconut_test["canonical_smiles"].notna()
].copy()

training = training[
    training["smiles"].notna()
].copy()

coconut_test["canonical_smiles"] = (
    coconut_test["canonical_smiles"].astype(str)
)

training["smiles"] = (
    training["smiles"].astype(str)
)

print(
    f"\nCOCONUT test after missing SMILES removal: "
    f"{len(coconut_test):,}"
)

print(
    f"Training after missing SMILES removal: "
    f"{len(training):,}"
)


# ============================================================
# 3. REMOVE DUPLICATE COCONUT SMILES
# ============================================================

before = len(coconut_test)

coconut_test = (
    coconut_test
    .drop_duplicates(subset="canonical_smiles")
    .reset_index(drop=True)
)

print(
    f"Removed {before - len(coconut_test):,} "
    f"duplicate COCONUT compounds"
)


# ============================================================
# 4. GENERATE TRAINING FINGERPRINTS
# ============================================================

def get_morgan_fp(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=True,
        useBondTypes=True
    )


print("\nGenerating training fingerprints...")

training_fps = []

valid_training_indices = []

for i, smiles in enumerate(
    tqdm(training["smiles"], desc="Training")
):

    fp = get_morgan_fp(smiles)

    if fp is not None:
        training_fps.append(fp)
        valid_training_indices.append(i)


training = (
    training
    .iloc[valid_training_indices]
    .reset_index(drop=True)
)

print(
    f"Valid training compounds: "
    f"{len(training):,}"
)


# ============================================================
# 5. GENERATE COCONUT TEST FINGERPRINTS
# ============================================================

print("\nGenerating COCONUT test fingerprints...")

coconut_fps = []

valid_coconut_indices = []

for i, smiles in enumerate(
    tqdm(
        coconut_test["canonical_smiles"],
        desc="COCONUT test"
    )
):

    fp = get_morgan_fp(smiles)

    if fp is not None:
        coconut_fps.append(fp)
        valid_coconut_indices.append(i)


coconut_test = (
    coconut_test
    .iloc[valid_coconut_indices]
    .reset_index(drop=True)
)

print(
    f"Valid COCONUT test compounds: "
    f"{len(coconut_test):,}"
)


# ============================================================
# 6. CALCULATE TANIMOTO SIMILARITIES
# ============================================================

print("\nStarting Tanimoto calculations...")

total_comparisons = (
    len(coconut_fps) *
    len(training_fps)
)

print(
    f"Total Tanimoto comparisons: "
    f"{total_comparisons:,}"
)


max_tanimoto = []
top5_mean_tanimoto = []

num_ge_060 = []
num_ge_070 = []
num_ge_080 = []

nearest_training_index = []


start = time.time()


for fp in tqdm(
    coconut_fps,
    desc="Tanimoto"
):

    similarities = (
        DataStructs.BulkTanimotoSimilarity(
            fp,
            training_fps
        )
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Maximum similarity
    # --------------------------------------------------------

    nearest_idx = int(
        np.argmax(similarities)
    )

    max_similarity = float(
        similarities[nearest_idx]
    )


    # --------------------------------------------------------
    # Top 5 similarity mean
    # --------------------------------------------------------

    top5 = np.partition(
        similarities,
        -5
    )[-5:]

    top5_mean = float(
        np.mean(top5)
    )


    # --------------------------------------------------------
    # Counts above thresholds
    # --------------------------------------------------------

    count_060 = int(
        np.sum(similarities >= 0.60)
    )

    count_070 = int(
        np.sum(similarities >= 0.70)
    )

    count_080 = int(
        np.sum(similarities >= 0.80)
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    max_tanimoto.append(
        max_similarity
    )

    top5_mean_tanimoto.append(
        top5_mean
    )

    num_ge_060.append(
        count_060
    )

    num_ge_070.append(
        count_070
    )

    num_ge_080.append(
        count_080
    )

    nearest_training_index.append(
        nearest_idx
    )


elapsed = time.time() - start


print("\nFinished!")

print(
    f"Time elapsed: "
    f"{elapsed:.2f} seconds"
)

print(
    f"Time elapsed: "
    f"{elapsed / 60:.2f} minutes"
)


# ============================================================
# 7. ADD RESULTS TO DATAFRAME
# ============================================================

coconut_test["max_tanimoto_training"] = (
    max_tanimoto
)

coconut_test["top5_mean_tanimoto_training"] = (
    top5_mean_tanimoto
)

coconut_test["num_training_ge_0.60"] = (
    num_ge_060
)

coconut_test["num_training_ge_0.70"] = (
    num_ge_070
)

coconut_test["num_training_ge_0.80"] = (
    num_ge_080
)

coconut_test["nearest_training_index"] = (
    nearest_training_index
)


# ============================================================
# 8. GET NEAREST TRAINING SMILES
# ============================================================

coconut_test["nearest_training_smiles"] = [
    training.iloc[i]["smiles"]
    for i in nearest_training_index
]


# ============================================================
# 9. APPLICABILITY CATEGORY
# ============================================================

def categorize_similarity(x):

    if x >= 0.80:
        return "Very high"

    elif x >= 0.70:
        return "High"

    elif x >= 0.60:
        return "Moderate"

    elif x >= 0.50:
        return "Low"

    else:
        return "Very low"


coconut_test["applicability_category"] = (
    coconut_test[
        "max_tanimoto_training"
    ].apply(categorize_similarity)
)


# ============================================================
# 10. PRINT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TANIMOTO SUMMARY")
print("=" * 60)

print(
    coconut_test[
        "max_tanimoto_training"
    ].describe()
)

print("\nApplicability categories:")

print(
    coconut_test[
        "applicability_category"
    ].value_counts()
)


# ============================================================
# 11. SAVE TEST RESULTS
# ============================================================

output = (
    "coconut_test_tanimoto_results.csv"
)

coconut_test.to_csv(
    output,
    index=False
)

print(f"\nSaved to: {output}")

COCONUT test compounds: 10,000
Training compounds: 5,779


AssertionError: 

In [ ]:
# ============================================================
# FULL COCONUT (~700k) vs JAK2 QSAR TRAINING SET
# PARALLEL TANIMOTO ANALYSIS — GOOGLE COLAB
# ============================================================

import pandas as pd
import numpy as np
import multiprocessing as mp
import time
import gc

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from tqdm.auto import tqdm


# ============================================================
# 1. SETTINGS
# ============================================================

N_WORKERS = min(4, mp.cpu_count())

# Process COCONUT in batches to control RAM usage
BATCH_SIZE = 10_000

print("=" * 60)
print("SETTINGS")
print("=" * 60)

print(f"Available CPUs: {mp.cpu_count()}")
print(f"Workers: {N_WORKERS}")
print(f"Batch size: {BATCH_SIZE:,}")


# ============================================================
# 2. CHECK DATA
# ============================================================

print("\n" + "=" * 60)
print("DATA")
print("=" * 60)

print(f"COCONUT compounds: {len(coconut):,}")
print(f"Training compounds: {len(training):,}")

assert "canonical_smiles" in coconut.columns
assert "smiles" in training.columns


# ============================================================
# 3. CLEAN SMILES
# ============================================================

coconut = coconut[
    coconut["canonical_smiles"].notna()
].copy()

training = training[
    training["smiles"].notna()
].copy()

coconut["canonical_smiles"] = (
    coconut["canonical_smiles"].astype(str)
)

training["smiles"] = (
    training["smiles"].astype(str)
)


# ============================================================
# 4. REMOVE DUPLICATE COCONUT SMILES
# ============================================================

before = len(coconut)

coconut = (
    coconut
    .drop_duplicates(
        subset="canonical_smiles"
    )
    .reset_index(drop=True)
)

print(
    f"\nRemoved {before - len(coconut):,} "
    f"duplicate COCONUT SMILES"
)

print(
    f"Unique COCONUT compounds: "
    f"{len(coconut):,}"
)


# ============================================================
# 5. MORGAN FINGERPRINT FUNCTION
# ============================================================

def get_morgan_fp(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=True,
        useBondTypes=True
    )


# ============================================================
# 6. TRAINING FINGERPRINTS
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TRAINING FINGERPRINTS")
print("=" * 60)

training_fps = []
valid_training_indices = []

start = time.time()

for i, smiles in enumerate(
    tqdm(
        training["smiles"],
        desc="Training"
    )
):

    fp = get_morgan_fp(smiles)

    if fp is not None:

        training_fps.append(fp)

        valid_training_indices.append(i)


training = (
    training
    .iloc[valid_training_indices]
    .reset_index(drop=True)
)

print(
    f"\nValid training compounds: "
    f"{len(training):,}"
)

print(
    f"Time: "
    f"{time.time() - start:.2f} sec"
)


# ============================================================
# 7. GLOBAL TRAINING FINGERPRINTS
# ============================================================

TRAINING_FPS = training_fps


# ============================================================
# 8. WORKER FUNCTION
# ============================================================

def calculate_tanimoto(fp):

    similarities = DataStructs.BulkTanimotoSimilarity(
        fp,
        TRAINING_FPS
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float32
    )


    # Maximum similarity
    nearest_idx = int(
        np.argmax(similarities)
    )

    max_similarity = float(
        similarities[nearest_idx]
    )


    # Top 5 mean
    top5 = np.partition(
        similarities,
        -5
    )[-5:]

    top5_mean = float(
        np.mean(top5)
    )


    # Similarity counts
    count_060 = int(
        np.sum(similarities >= 0.60)
    )

    count_070 = int(
        np.sum(similarities >= 0.70)
    )

    count_080 = int(
        np.sum(similarities >= 0.80)
    )


    return (
        max_similarity,
        top5_mean,
        count_060,
        count_070,
        count_080,
        nearest_idx
    )


# ============================================================
# 9. PREALLOCATE RESULT ARRAYS
# ============================================================

n_coconut = len(coconut)

max_tanimoto = np.full(
    n_coconut,
    np.nan,
    dtype=np.float32
)

top5_mean_tanimoto = np.full(
    n_coconut,
    np.nan,
    dtype=np.float32
)

num_ge_060 = np.zeros(
    n_coconut,
    dtype=np.int32
)

num_ge_070 = np.zeros(
    n_coconut,
    dtype=np.int32
)

num_ge_080 = np.zeros(
    n_coconut,
    dtype=np.int32
)

nearest_training_index = np.full(
    n_coconut,
    -1,
    dtype=np.int32
)


# ============================================================
# 10. PROCESS COCONUT IN BATCHES
# ============================================================

print("\n" + "=" * 60)
print("STARTING FULL COCONUT TANIMOTO ANALYSIS")
print("=" * 60)

total_comparisons = (
    n_coconut *
    len(training_fps)
)

print(
    f"COCONUT compounds: "
    f"{n_coconut:,}"
)

print(
    f"Training compounds: "
    f"{len(training_fps):,}"
)

print(
    f"Total comparisons: "
    f"{total_comparisons:,}"
)

print(
    f"Workers: "
    f"{N_WORKERS}"
)

print(
    f"Batch size: "
    f"{BATCH_SIZE:,}"
)


overall_start = time.time()


# ------------------------------------------------------------
# Process batches
# ------------------------------------------------------------

for start_idx in range(
    0,
    n_coconut,
    BATCH_SIZE
):

    end_idx = min(
        start_idx + BATCH_SIZE,
        n_coconut
    )

    batch_smiles = coconut[
        "canonical_smiles"
    ].iloc[
        start_idx:end_idx
    ]


    # --------------------------------------------------------
    # Generate fingerprints
    # --------------------------------------------------------

    batch_fps = []

    batch_indices = []

    for local_idx, smiles in enumerate(
        batch_smiles
    ):

        fp = get_morgan_fp(smiles)

        if fp is not None:

            batch_fps.append(fp)

            batch_indices.append(
                start_idx + local_idx
            )


    # --------------------------------------------------------
    # Parallel Tanimoto
    # --------------------------------------------------------

    with mp.Pool(
        processes=N_WORKERS
    ) as pool:

        results = list(
            pool.imap(
                calculate_tanimoto,
                batch_fps,
                chunksize=100
            )
        )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    for local_result_idx, result in enumerate(
        results
    ):

        global_idx = (
            batch_indices[local_result_idx]
        )

        (
            max_similarity,
            top5_mean,
            count_060,
            count_070,
            count_080,
            nearest_idx
        ) = result


        max_tanimoto[
            global_idx
        ] = max_similarity

        top5_mean_tanimoto[
            global_idx
        ] = top5_mean

        num_ge_060[
            global_idx
        ] = count_060

        num_ge_070[
            global_idx
        ] = count_070

        num_ge_080[
            global_idx
        ] = count_080

        nearest_training_index[
            global_idx
        ] = nearest_idx


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = end_idx

    elapsed = (
        time.time() -
        overall_start
    )

    rate = (
        processed / elapsed
    )

    remaining = (
        n_coconut -
        processed
    )

    eta_seconds = (
        remaining / rate
        if rate > 0
        else np.nan
    )


    print(
        f"\nProcessed "
        f"{processed:,}/{n_coconut:,} "
        f"({100 * processed / n_coconut:.1f}%)"
    )

    print(
        f"Rate: "
        f"{rate:.1f} compounds/sec"
    )

    print(
        f"Estimated remaining: "
        f"{eta_seconds / 60:.1f} min"
    )


    # --------------------------------------------------------
    # Free batch memory
    # --------------------------------------------------------

    del batch_fps
    del results

    gc.collect()


# ============================================================
# 11. FINAL RUNTIME
# ============================================================

total_time = (
    time.time() -
    overall_start
)

print("\n" + "=" * 60)
print("TANIMOTO ANALYSIS COMPLETE")
print("=" * 60)

print(
    f"Total time: "
    f"{total_time / 60:.2f} minutes"
)


# ============================================================
# 12. ADD RESULTS TO COCONUT
# ============================================================

coconut[
    "max_tanimoto_training"
] = max_tanimoto

coconut[
    "top5_mean_tanimoto_training"
] = top5_mean_tanimoto

coconut[
    "num_training_ge_0.60"
] = num_ge_060

coconut[
    "num_training_ge_0.70"
] = num_ge_070

coconut[
    "num_training_ge_0.80"
] = num_ge_080

coconut[
    "nearest_training_index"
] = nearest_training_index


# ============================================================
# 13. NEAREST TRAINING SMILES
# ============================================================

nearest_smiles = []

for idx in nearest_training_index:

    if idx >= 0:

        nearest_smiles.append(
            training.iloc[int(idx)]["smiles"]
        )

    else:

        nearest_smiles.append(
            None
        )


coconut[
    "nearest_training_smiles"
] = nearest_smiles


# ============================================================
# 14. APPLICABILITY CATEGORY
# ============================================================

def categorize_similarity(x):

    if pd.isna(x):
        return "Invalid"

    elif x >= 0.80:
        return "Very high"

    elif x >= 0.70:
        return "High"

    elif x >= 0.60:
        return "Moderate"

    elif x >= 0.50:
        return "Low"

    else:
        return "Very low"


coconut[
    "applicability_category"
] = (
    coconut[
        "max_tanimoto_training"
    ].apply(
        categorize_similarity
    )
)


# ============================================================
# 15. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("MAX TANIMOTO SUMMARY")
print("=" * 60)

print(
    coconut[
        "max_tanimoto_training"
    ].describe()
)


print("\n" + "=" * 60)
print("APPLICABILITY CATEGORIES")
print("=" * 60)

print(
    coconut[
        "applicability_category"
    ].value_counts()
)


# ============================================================
# 16. TOP 20 MOST SIMILAR
# ============================================================

print("\n" + "=" * 60)
print("TOP 20 MOST SIMILAR COCONUT COMPOUNDS")
print("=" * 60)

print(
    coconut[
        [
            "canonical_smiles",
            "max_tanimoto_training",
            "top5_mean_tanimoto_training",
            "nearest_training_smiles"
        ]
    ]
    .sort_values(
        "max_tanimoto_training",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 17. SAVE RESULTS
# ============================================================

output_file = (
    "/content/coconut_full_tanimoto_results.csv"
)

coconut.to_csv(
    output_file,
    index=False
)


print("\n" + "=" * 60)
print("RESULTS SAVED")
print("=" * 60)

print(output_file)

SETTINGS
Available CPUs: 2
Workers: 2
Batch size: 10,000

DATA
COCONUT compounds: 738,827
Training compounds: 5,779

Removed 0 duplicate COCONUT SMILES
Unique COCONUT compounds: 738,827

GENERATING TRAINING FINGERPRINTS


Training:   0%|          | 0/5779 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:18:08] DEPRECATION WARNING: please use MorganGenerator
[21:1


Valid training compounds: 5,779
Time: 2.49 sec

STARTING FULL COCONUT TANIMOTO ANALYSIS
COCONUT compounds: 738,827
Training compounds: 5,779
Total comparisons: 4,269,681,233
Workers: 2
Batch size: 10,000


Streaming output truncated to the last 5000 lines.
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:18:12] DEPRECATION WARNING: please use MorganGenerator
[21:1


Processed 10,000/738,827 (1.4%)
Rate: 444.2 compounds/sec
Estimated remaining: 27.3 min


... (142 outputs left out)

Streaming output truncated to the last 5000 lines.
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:46:18] DEPRECATION WARNING: please use MorganGenerator
[21:4


Processed 730,000/738,827 (98.8%)
Rate: 426.3 compounds/sec
Estimated remaining: 0.3 min


Streaming output truncated to the last 5000 lines.
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:46:45] DEPRECATION WARNING: please use MorganGenerator
[21:4


Processed 738,827/738,827 (100.0%)
Rate: 425.9 compounds/sec
Estimated remaining: 0.0 min

TANIMOTO ANALYSIS COMPLETE
Total time: 28.91 minutes

MAX TANIMOTO SUMMARY
count    738823.000000
mean          0.183171
std           0.043104
min           0.000000
25%           0.153226
50%           0.173913
75%           0.205882
max           1.000000
Name: max_tanimoto_training, dtype: float64

APPLICABILITY CATEGORIES
applicability_category
Very low     738681
Low              60
Moderate         44
High             28
Very high        10
Invalid           4
Name: count, dtype: int64

TOP 20 MOST SIMILAR COCONUT COMPOUNDS
                                                                              canonical_smiles  max_tanimoto_training  top5_mean_tanimoto_training                                                        nearest_training_smiles
         CN[C@@H]1C[C@H]2O[C@@](C)([C@@H]1OC)N1C3=CC=CC=C3C3=C4CNC(=O)C4=C4C5=CC=CC=C5N2C4=C31               1.000000                     0.43335

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# EXACT QSAR MORGAN GENERATOR
# ============================================================

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=True,
    useBondTypes=True
)


# ============================================================
# GENERATE FINGERPRINTS
# ============================================================

fps = []
valid_indices = []

for i, smiles in enumerate(
    tqdm(training["smiles"], desc="Generating Morgan fingerprints")
):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        continue

    fp = morgan_gen.GetFingerprint(mol)

    fps.append(fp)
    valid_indices.append(i)


print(f"Valid compounds: {len(fps):,}")


# ============================================================
# NEAREST-NEIGHBOR TANIMOTO
# ============================================================

nearest_similarity = []

for i, fp in enumerate(
    tqdm(fps, desc="Calculating nearest-neighbor similarity")
):

    similarities = DataStructs.BulkTanimotoSimilarity(
        fp,
        fps
    )

    similarities = np.asarray(
        similarities,
        dtype=np.float32
    )

    # Exclude itself
    similarities[i] = -1

    nearest_similarity.append(
        np.max(similarities)
    )


nearest_similarity = np.asarray(
    nearest_similarity,
    dtype=np.float32
)


# ============================================================
# RESULTS
# ============================================================

training_similarity = (
    training
    .iloc[valid_indices]
    .copy()
    .reset_index(drop=True)
)

training_similarity[
    "nearest_jak_tanimoto"
] = nearest_similarity


print("\n" + "=" * 60)
print("JAK INHIBITOR CHEMICAL SIMILARITY")
print("=" * 60)

print(
    training_similarity[
        "nearest_jak_tanimoto"
    ].describe()
)

print(
    "\nMedian:",
    np.median(nearest_similarity)
)

Generating Morgan fingerprints:   0%|          | 0/5779 [00:00<?, ?it/s]

Valid compounds: 5,779


Calculating nearest-neighbor similarity:   0%|          | 0/5779 [00:00<?, ?it/s]


JAK INHIBITOR CHEMICAL SIMILARITY
count    5779.000000
mean        0.842009
std         0.080351
min         0.233766
25%         0.797297
50%         0.841463
75%         0.885714
max         1.000000
Name: nearest_jak_tanimoto, dtype: float64

Median: 0.8414634


In [ ]:
full_jak2 = pd.read_csv("chembl_jak2.tsv", sep = '\t')

/tmp/ipykernel_1202/3396564329.py:1: DtypeWarning: Columns (27,28,30) have mixed types. Specify dtype option on import or set low_memory=False.
  full_jak2 = pd.read_csv("chembl_jak2.tsv", sep = '\t')


In [ ]:
coconut_lite = pd.read_csv("/content/coconut_csv_lite-08-2026.csv")

/tmp/ipykernel_1202/2627071268.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut_lite = pd.read_csv("/content/coconut_csv_lite-08-2026.csv")


In [ ]:
# ============================================================
# COCONUT LITE vs FULL ChEMBL JAK2 SET
# Exact QSAR Morgan fingerprint
# ============================================================

import numpy as np
import pandas as pd
import multiprocessing as mp
import time
import gc

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from tqdm.auto import tqdm


# ============================================================
# 1. SETTINGS
# ============================================================

N_WORKERS = min(4, mp.cpu_count())
BATCH_SIZE = 5000

print("=" * 60)
print("SETTINGS")
print("=" * 60)

print(f"Available CPUs: {mp.cpu_count()}")
print(f"Workers: {N_WORKERS}")
print(f"Batch size: {BATCH_SIZE:,}")


# ============================================================
# 2. CHECK DATA
# ============================================================

print("\n" + "=" * 60)
print("DATA")
print("=" * 60)

print(f"Full JAK2 compounds: {len(full_jak2):,}")
print(f"COCONUT Lite compounds: {len(coconut_lite):,}")

assert "Smiles" in full_jak2.columns
assert "canonical_smiles" in coconut_lite.columns


# ============================================================
# 3. CHECK INVALID / MISSING SMILES
# ============================================================

print("\n" + "=" * 60)
print("SMILES QUALITY CHECK")
print("=" * 60)

jak2_missing = full_jak2["Smiles"].isna().sum()

jak2_nonstring = (
    ~full_jak2["Smiles"]
    .apply(lambda x: isinstance(x, str))
).sum()

coconut_missing = (
    coconut_lite["canonical_smiles"]
    .isna()
    .sum()
)

coconut_nonstring = (
    ~coconut_lite["canonical_smiles"]
    .apply(lambda x: isinstance(x, str))
).sum()

print(f"JAK2 missing SMILES: {jak2_missing:,}")
print(f"JAK2 non-string SMILES: {jak2_nonstring:,}")

print(f"COCONUT missing SMILES: {coconut_missing:,}")
print(f"COCONUT non-string SMILES: {coconut_nonstring:,}")


# ============================================================
# 4. EXACT QSAR MORGAN GENERATOR
#
# Morgan / ECFP4
# Radius = 2
# 2048 bits
# Chirality = True
# Bond types = True
# Standard RDKit atom invariants
# Feature invariants = OFF
# Count simulation = OFF
# ============================================================

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=True,
    useBondTypes=True
)


# ============================================================
# 5. FINGERPRINT FUNCTION
# ============================================================

def get_morgan_fp(smiles):

    # Skip missing / non-string values
    if not isinstance(smiles, str):
        return None

    # Skip empty strings
    if not smiles.strip():
        return None

    # Convert SMILES to RDKit molecule
    mol = Chem.MolFromSmiles(smiles)

    # Skip invalid SMILES
    if mol is None:
        return None

    # Exact binary Morgan fingerprint used by QSAR
    return morgan_gen.GetFingerprint(mol)


# ============================================================
# 6. GENERATE FULL JAK2 FINGERPRINTS
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FULL JAK2 FINGERPRINTS")
print("=" * 60)

jak2_fps = []
valid_jak2_indices = []

start = time.time()

for i, smiles in enumerate(
    tqdm(
        full_jak2["Smiles"],
        desc="Full JAK2"
    )
):

    fp = get_morgan_fp(smiles)

    if fp is not None:
        jak2_fps.append(fp)
        valid_jak2_indices.append(i)


# Keep only compounds with valid fingerprints

full_jak2 = (
    full_jak2
    .iloc[valid_jak2_indices]
    .reset_index(drop=True)
)


print(
    f"\nValid JAK2 compounds: "
    f"{len(jak2_fps):,}"
)

print(
    f"Invalid/skipped JAK2 compounds: "
    f"{len(full_jak2) - len(jak2_fps):,}"
)

print(
    f"Time: "
    f"{time.time() - start:.2f} sec"
)


# ============================================================
# 7. GLOBAL REFERENCE FINGERPRINTS
# ============================================================

REFERENCE_FPS = jak2_fps


# ============================================================
# 8. WORKER FUNCTION
# ============================================================

def calculate_max_tanimoto(fp):

    similarities = DataStructs.BulkTanimotoSimilarity(
        fp,
        REFERENCE_FPS
    )

    return float(max(similarities))


# ============================================================
# 9. PREALLOCATE RESULTS
# ============================================================

n_coconut = len(coconut_lite)

max_tanimoto = np.full(
    n_coconut,
    np.nan,
    dtype=np.float32
)


# ============================================================
# 10. PROCESS COCONUT LITE IN BATCHES
# ============================================================

print("\n" + "=" * 60)
print("COCONUT → FULL JAK2 SIMILARITY")
print("=" * 60)

total_comparisons = (
    n_coconut *
    len(REFERENCE_FPS)
)

print(
    f"Total theoretical comparisons: "
    f"{total_comparisons:,}"
)

overall_start = time.time()


for start_idx in range(
    0,
    n_coconut,
    BATCH_SIZE
):

    end_idx = min(
        start_idx + BATCH_SIZE,
        n_coconut
    )


    # --------------------------------------------------------
    # COCONUT SMILES
    # --------------------------------------------------------

    batch_smiles = (
        coconut_lite[
            "canonical_smiles"
        ]
        .iloc[
            start_idx:end_idx
        ]
    )


    # --------------------------------------------------------
    # Generate fingerprints
    # --------------------------------------------------------

    batch_fps = []
    batch_indices = []

    for local_idx, smiles in enumerate(
        batch_smiles
    ):

        fp = get_morgan_fp(smiles)

        if fp is not None:

            batch_fps.append(fp)

            batch_indices.append(
                start_idx + local_idx
            )


    # --------------------------------------------------------
    # Parallel Tanimoto calculation
    # --------------------------------------------------------

    with mp.Pool(
        processes=N_WORKERS
    ) as pool:

        results = list(
            tqdm(
                pool.imap(
                    calculate_max_tanimoto,
                    batch_fps,
                    chunksize=100
                ),
                total=len(batch_fps),
                desc=f"{start_idx:,}-{end_idx:,}",
                leave=False
            )
        )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    for local_idx, similarity in enumerate(
        results
    ):

        global_idx = batch_indices[
            local_idx
        ]

        max_tanimoto[
            global_idx
        ] = similarity


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = end_idx

    elapsed = (
        time.time() -
        overall_start
    )

    rate = (
        processed / elapsed
    )

    remaining = (
        n_coconut -
        processed
    )

    eta = (
        remaining / rate
        if rate > 0
        else np.nan
    )

    print(
        f"Processed "
        f"{processed:,}/{n_coconut:,} "
        f"({100 * processed/n_coconut:.1f}%)"
    )

    print(
        f"Rate: "
        f"{rate:.1f} compounds/sec"
    )

    print(
        f"ETA: "
        f"{eta/60:.1f} min"
    )


    # Free memory

    del batch_fps
    del results

    gc.collect()


# ============================================================
# 11. ADD RESULTS TO COCONUT DATAFRAME
# ============================================================

coconut_lite[
    "max_tanimoto_full_jak2"
] = max_tanimoto


# ============================================================
# 12. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(
    coconut_lite[
        "max_tanimoto_full_jak2"
    ].describe()
)


# ============================================================
# 13. THRESHOLD COUNTS
# ============================================================

print("\n" + "=" * 60)
print("SIMILARITY THRESHOLDS")
print("=" * 60)

for threshold in [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    count = np.sum(
        max_tanimoto >= threshold
    )

    percentage = (
        100 * count / n_coconut
    )

    print(
        f"Tanimoto >= {threshold:.2f}: "
        f"{count:,} "
        f"({percentage:.3f}%)"
    )


# ============================================================
# 14. TOP 20 MOST SIMILAR COMPOUNDS
# ============================================================

print("\n" + "=" * 60)
print("TOP 20 MOST SIMILAR COCONUT COMPOUNDS")
print("=" * 60)

print(
    coconut_lite[
        [
            "canonical_smiles",
            "max_tanimoto_full_jak2"
        ]
    ]
    .sort_values(
        "max_tanimoto_full_jak2",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 15. SAVE RESULTS
# ============================================================

output_file = (
    "/content/"
    "coconut_lite_vs_full_jak2_tanimoto.csv"
)

coconut_lite.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("SAVED")
print("=" * 60)

print(output_file)

SETTINGS
Available CPUs: 2
Workers: 2
Batch size: 5,000

DATA
Full JAK2 compounds: 56,211
COCONUT Lite compounds: 738,827

SMILES QUALITY CHECK
JAK2 missing SMILES: 0
JAK2 non-string SMILES: 0
COCONUT missing SMILES: 0
COCONUT non-string SMILES: 0

GENERATING FULL JAK2 FINGERPRINTS


Full JAK2:   0%|          | 0/56211 [00:00<?, ?it/s]


Valid JAK2 compounds: 56,211
Invalid/skipped JAK2 compounds: 0
Time: 25.47 sec

COCONUT → FULL JAK2 SIMILARITY
Total theoretical comparisons: 41,530,204,497


0-5,000:   0%|          | 0/5000 [00:00<?, ?it/s]

Processed 5,000/738,827 (0.7%)
Rate: 61.0 compounds/sec
ETA: 200.5 min


5,000-10,000:   0%|          | 0/5000 [00:00<?, ?it/s]

... (303 outputs left out)

730,000-735,000:   0%|          | 0/5000 [00:00<?, ?it/s]

Processed 735,000/738,827 (99.5%)
Rate: 60.4 compounds/sec
ETA: 1.1 min


735,000-738,827:   0%|          | 0/3827 [00:00<?, ?it/s]

Processed 738,827/738,827 (100.0%)
Rate: 60.4 compounds/sec
ETA: 0.0 min

FINAL RESULTS
count    738823.000000
mean          0.238835
std           0.074243
min           0.019231
25%           0.186667
50%           0.223684
75%           0.275000
max           1.000000
Name: max_tanimoto_full_jak2, dtype: float64

SIMILARITY THRESHOLDS
Tanimoto >= 0.10: 738,241 (99.921%)
Tanimoto >= 0.15: 712,171 (96.392%)
Tanimoto >= 0.20: 484,791 (65.616%)
Tanimoto >= 0.25: 263,334 (35.642%)
Tanimoto >= 0.30: 125,020 (16.921%)
Tanimoto >= 0.40: 23,795 (3.221%)
Tanimoto >= 0.50: 7,208 (0.976%)
Tanimoto >= 0.60: 2,355 (0.319%)
Tanimoto >= 0.70: 789 (0.107%)
Tanimoto >= 0.80: 334 (0.045%)
Tanimoto >= 0.90: 212 (0.029%)

TOP 20 MOST SIMILAR COCONUT COMPOUNDS
                                                          canonical_smiles  max_tanimoto_full_jak2
                     COC1=CC(/C=C(\C#N)C(N)=O)=CC(CSC2=NC3=CC=CC=C3S2)=C1O                     1.0
                             COC1=CC=C(C2=NN(C3CCC

In [ ]:
import multiprocessing as mp

print("CPU cores available:", mp.cpu_count())

CPU cores available: 2


In [ ]:
# ============================================================
# JAK2 → JAK2 NEAREST-NEIGHBOR Tanimoto
#
# For every JAK2 inhibitor:
#   find the most similar OTHER JAK2 inhibitor
#
# Uses the exact QSAR Morgan fingerprint:
#   Radius = 2
#   2048 bits
#   Chirality = True
#   Bond types = True
#   Binary fingerprint
# ============================================================

import numpy as np
import pandas as pd
import time
from rdkit import DataStructs
from tqdm.auto import tqdm


# ============================================================
# 1. CHECK FINGERPRINTS
# ============================================================

print("=" * 60)
print("JAK2 NEAREST-NEIGHBOR ANALYSIS")
print("=" * 60)

print(
    f"Number of JAK2 fingerprints: "
    f"{len(jak2_fps):,}"
)


# ============================================================
# 2. CALCULATE NEAREST OTHER JAK2 COMPOUND
# ============================================================

nearest_jak2_similarity = np.full(
    len(jak2_fps),
    np.nan,
    dtype=np.float32
)

nearest_jak2_index = np.full(
    len(jak2_fps),
    -1,
    dtype=np.int32
)


start = time.time()


for i, fp in enumerate(
    tqdm(
        jak2_fps,
        desc="JAK2 nearest neighbors"
    )
):

    similarities = DataStructs.BulkTanimotoSimilarity(
        fp,
        jak2_fps
    )

    # Remove self-similarity
    similarities[i] = -1.0

    # Find most similar OTHER compound
    nearest_index = int(
        np.argmax(similarities)
    )

    nearest_jak2_similarity[i] = (
        similarities[nearest_index]
    )

    nearest_jak2_index[i] = nearest_index


elapsed = time.time() - start


# ============================================================
# 3. ADD RESULTS TO DATAFRAME
# ============================================================

full_jak2[
    "nearest_jak2_tanimoto"
] = nearest_jak2_similarity

full_jak2[
    "nearest_jak2_index"
] = nearest_jak2_index


# ============================================================
# 4. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("JAK2 → JAK2 NEAREST-NEIGHBOR SUMMARY")
print("=" * 60)

print(
    full_jak2[
        "nearest_jak2_tanimoto"
    ].describe()
)

print(
    f"\nCalculation time: "
    f"{elapsed / 60:.2f} minutes"
)


# ============================================================
# 5. THRESHOLD ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("SIMILARITY THRESHOLDS")
print("=" * 60)

for threshold in [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    count = np.sum(
        nearest_jak2_similarity >= threshold
    )

    percentage = (
        100 *
        count /
        len(jak2_fps)
    )

    print(
        f"Tanimoto >= {threshold:.2f}: "
        f"{count:,} "
        f"({percentage:.3f}%)"
    )


# ============================================================
# 6. TOP 20 MOST CLUSTERED JAK2 COMPOUNDS
# ============================================================

print("\n" + "=" * 60)
print("20 JAK2 COMPOUNDS WITH MOST SIMILAR NEIGHBORS")
print("=" * 60)

print(
    full_jak2[
        [
            "Smiles",
            "nearest_jak2_tanimoto",
            "nearest_jak2_index"
        ]
    ]
    .sort_values(
        "nearest_jak2_tanimoto",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 7. SAVE
# ============================================================

output_file = (
    "/content/"
    "full_jak2_nearest_neighbor_tanimoto.csv"
)

full_jak2.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)

ModuleNotFoundError: No module named 'rdkit'